# Modélisation du Système de Recommandation
Dans ce notebook, nous allons développer et comparer deux approches pour notre système de recommandation :
1. **Content-Based (Filtrage basé sur le contenu)**
2. **Collaborative Filtering (Filtrage collaboratif)**

L'objectif est de créer une fonction prenant en entrée un `user_id` et retournant 5 `article_id` recommandés.

## 1. Importation des bibliothèques et chargement des données

In [5]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("Librairies importées avec succès.")

Librairies importées avec succès.


In [6]:
# Définition des chemins
DATA_DIR = "../Data/news-portal-user-interactions-by-globocom"
GENERATED_DIR = "../Generated"

METADATA_PATH = os.path.join(DATA_DIR, "articles_metadata.csv")
CLICKS_SAMPLE_PATH = os.path.join(DATA_DIR, "clicks_sample.csv")
EMBEDDINGS_PCA_PATH = os.path.join(GENERATED_DIR, "articles_embeddings_pca.pickle")

# Chargement des données
print("Chargement des métadonnées des articles...")
articles_df = pd.read_csv(METADATA_PATH)

print("Chargement de l'historique des clics...")
clicks_df = pd.read_csv(CLICKS_SAMPLE_PATH)

print("Chargement des embeddings réduits par ACP...")
with open(EMBEDDINGS_PCA_PATH, 'rb') as f:
    embeddings_pca = pickle.load(f)

print("Chargement terminé !")
print(f"Dimensions des embeddings réduits : {embeddings_pca.shape}")

Chargement des métadonnées des articles...
Chargement de l'historique des clics...
Chargement des embeddings réduits par ACP...
Chargement terminé !
Dimensions des embeddings réduits : (364047, 52)


## 2. Approche Content-Based (Basée sur le contenu)
L'idée ici est de recommander des articles qui sont sémantiquement similaires à ceux que l'utilisateur a déjà lus.

**Étapes :**
1. Identifier les articles lus par un utilisateur cible (`user_id`).
2. Récupérer les embeddings de ces articles lus.
3. Calculer le 'profil' de l'utilisateur (par exemple, la moyenne des embeddings des articles qu'il a lus).
4. Calculer la similarité cosinus entre ce profil utilisateur et tous les autres articles disponibles.
5. Retourner les 5 articles les plus similaires (que l'utilisateur n'a pas encore lus).

In [7]:
def get_user_articles(user_id, clicks_df):
    """
    Retourne la liste des article_id lus par un utilisateur.
    """
    return clicks_df[clicks_df['user_id'] == user_id]['click_article_id'].tolist()

def recommend_content_based(user_id, clicks_df, embeddings, n_recommendations=5):
    """
    Génère des recommandations basées sur le contenu pour un utilisateur donné.
    """
    # 1. Obtenir les articles lus par l'utilisateur
    read_articles = get_user_articles(user_id, clicks_df)
    
    # 2. Gestion du Cold Start (utilisateur sans historique)
    if not read_articles:
        print(f"Utilisateur {user_id} sans historique. Renvoi des articles les plus populaires par défaut.")
        popular_articles = clicks_df['click_article_id'].value_counts().head(n_recommendations).index.tolist()
        return popular_articles
    
    # 3. Calculer le profil utilisateur (moyenne des embeddings des articles lus)
    # On s'assure que l'ID de l'article ne dépasse pas la taille de la matrice d'embeddings
    valid_articles = [art_id for art_id in read_articles if art_id < embeddings.shape[0]]
    
    if not valid_articles:
        return []
        
    user_profile = np.mean(embeddings[valid_articles], axis=0).reshape(1, -1)
    
    # 4. Calculer la similarité cosinus entre le profil et tous les embeddings
    similarities = cosine_similarity(user_profile, embeddings)[0]
    
    # 5. Trier et retourner le top N (en excluant les articles déjà lus)
    # Obtenir les indices triés par ordre décroissant de similarité
    similar_indices = np.argsort(similarities)[::-1]
    
    recommendations = []
    for idx in similar_indices:
        if idx not in read_articles:
            recommendations.append(int(idx))
        if len(recommendations) == n_recommendations:
            break
            
    return recommendations


In [8]:
def get_article_metadata(article_ids, articles_df):
    """
    Récupère les informations disponibles sur les articles (catégorie, nombre de mots, etc.).
    NOTE: Le dataset Globocom est anonymisé, il n'y a donc pas de vrais 'titres' ou 'libellés' d'articles.
    """
    return articles_df[articles_df['article_id'].isin(article_ids)]

# Test de la fonction de recommandation
sample_user_id = clicks_df['user_id'].iloc[0]
print(f"Recommandations pour l'utilisateur {sample_user_id} :")
recs = recommend_content_based(sample_user_id, clicks_df, embeddings_pca, n_recommendations=5)
print(f"IDs recommandés : {recs}\n")

print("Détails des articles recommandés (métadonnées) : ")
display(get_article_metadata(recs, articles_df))

Recommandations pour l'utilisateur 0 :
IDs recommandés : [157519, 157944, 159495, 156690, 162856]

Détails des articles recommandés (métadonnées) : 


,article_id,category_id,created_at_ts,publisher_id,words_count
156690,156690,281,1496859579000,0,170
157519,157519,281,1506064952000,0,272
157944,157944,281,1512504667000,0,204
159495,159495,281,1506021561000,0,261
162856,162856,281,1511000684000,0,237
